# 08주차 · 벡터 검색과 RAG

**임베딩 기반 데이터 과학 — 국립목포대학교 컴퓨터학부 4학년**

이 Notebook은 *Hands-On Large Language Models* 공개 저장소의 개념과 실습 흐름을
한국어 수업에 맞게 새로 구성한 파생 강의자료입니다. 원본은 Apache License 2.0을
따르며, 출처와 변경 사항은 `SOURCE_AND_LICENSE.md`에 기록했습니다.

- 원본: https://github.com/HandsOnLLM/Hands-On-Large-Language-Models
- 기준 커밋: `ea3390819997999a51983677b80b3aac4dc50ada`
- 권장 환경: Google Colab 또는 Python 3.11+


## 학습목표

- 수집·분할·임베딩·인덱싱·검색·생성을 분리한다.
- 문서 청크 크기와 중첩을 실험한다.
- 생성 답변에 근거 문서와 검색 점수를 표시한다.


In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom else 0.0


In [ ]:
documents = [
    {"id":"P01", "text":"청년 창업 지원 사업은 만 39세 이하 예비 창업자에게 교육과 사업화 자금을 제공한다."},
    {"id":"P02", "text":"섬 주민 원격진료 사업은 의료 접근성이 낮은 도서 지역 주민의 비대면 상담을 지원한다."},
    {"id":"P03", "text":"고령자 이동 지원 사업은 병원과 복지관 방문을 위한 예약형 차량을 운영한다."},
    {"id":"P04", "text":"해양 관광 콘텐츠 지원 사업은 지역의 섬과 항구를 활용한 관광 상품 개발 기업을 지원한다."},
]

def chunk_text(text, size=35, overlap=8):
    step = max(1, size-overlap)
    return [text[i:i+size] for i in range(0, len(text), step) if text[i:i+size].strip()]

chunks = []
for doc in documents:
    for j, chunk in enumerate(chunk_text(doc["text"])):
        chunks.append({"chunk_id": f"{doc['id']}-{j}", "doc_id": doc["id"], "text": chunk})
pd.DataFrame(chunks)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
corpus = [c["text"] for c in chunks]
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
X = vectorizer.fit_transform(corpus)

def retrieve(query, k=3):
    q = vectorizer.transform([query])
    scores = cosine_similarity(q, X).ravel()
    ids = np.argsort(-scores)[:k]
    return [{**chunks[i], "score": float(scores[i])} for i in ids]

hits = retrieve("청년이 창업 자금을 받을 수 있나요?")
pd.DataFrame(hits)


In [ ]:
def build_grounded_prompt(question, hits):
    evidence = "\n".join(f"[{h['chunk_id']}] {h['text']}" for h in hits)
    return (
        "다음 근거만 사용하여 질문에 답하세요.\n"
        "근거가 부족하면 '제공된 문서로 확인할 수 없습니다'라고 답하세요.\n"
        "문장 끝에 근거의 chunk_id를 표시하세요.\n\n"
        f"[근거]\n{evidence}\n\n[질문]\n{question}\n"
    )

print(build_grounded_prompt("청년이 창업 자금을 받을 수 있나요?", hits))


## 학생 활동

- 청크 크기 20·40·80을 비교하라.
- Dense 모델을 선택적으로 추가하되 TF-IDF 결과를 삭제하지 말라.
- 검색 실패와 생성 실패를 별도 열로 기록하라.
- API를 사용한다면 키를 Notebook에 저장하지 말고 환경변수를 사용하라.


---
## 학습 기록과 생성형 AI 사용 내역

다음 항목을 자신의 말로 작성하세요.

1. 이번 실습에서 가장 중요한 결과는 무엇인가?
2. 결과를 뒷받침하는 수치 또는 그래프는 무엇인가?
3. 실패하거나 예상과 달랐던 부분은 무엇인가?
4. 생성형 AI를 사용했다면 프롬프트, 채택·거부한 제안, 직접 검증한 내용을 기록하라.
